# DMRC RAG — 01. Setup & Retrieval Validation

This notebook only covers the **non-LLM** half of the pipeline:

```
clone repo -> install deps -> validate ChromaDB -> dense search -> hybrid search -> reranker
```

No model weights for Gemma are downloaded or loaded here, so this notebook runs fine on a
**CPU runtime** (Colab: Runtime -> Change runtime type -> CPU) and finishes in a couple of
minutes. Once everything below passes, move to **`02_Gemma_Inference_and_Serving.ipynb`**
for the LLM + API part, which does need a GPU runtime.

Split rationale: the original single notebook re-ran `pip install`/repo setup every time you
just wanted to test model inference, and mixed a slow one-time GPU load in with fast,
repeatable retrieval checks. Splitting means you only pay the GPU-session cost when you're
actually touching the LLM.

### 1. Clone the repo (idempotent)

In [3]:
%cd /content
!test -d dmrc && (echo "dmrc/ already present -- pulling latest" && cd dmrc && git pull) \
    || git clone https://github.com/sunvantaconsultancysolutions-design/dmrc.git
%cd /content/dmrc


/content
dmrc/ already present -- pulling latest
Already up to date.
/content/dmrc


### 2. Install dependencies

One install, matching `requirements.txt` exactly -- no separate torch install/uninstall loop needed on a fresh Colab CPU runtime.

In [2]:
!grep -v -E "^(bitsandbytes|nvidia-nvjitlink-cu13)" requirements.txt > /tmp/requirements_core.txt
!pip install -q -r /tmp/requirements_core.txt

In [ ]:
print("Restarting the kernel so numpy loads cleanly after the install above...")
print("Colab will auto-reconnect in a few seconds -- then continue running from the next cell.")
import os
os.kill(os.getpid(), 9)

### 3. Sanity-check the environment

One consolidated check instead of several repeated `import torch` cells.

In [1]:
import torch, transformers, sentence_transformers, chromadb

print(f"torch               : {torch.__version__}")
print(f"CUDA available      : {torch.cuda.is_available()}")
print(f"transformers        : {transformers.__version__}")
print(f"sentence_transformers: {sentence_transformers.__version__}")
print(f"chromadb            : {chromadb.__version__}")


torch               : 2.10.0+cu128
CUDA available      : True
transformers        : 4.57.6
sentence_transformers: 3.0.1
chromadb            : 0.5.5


### 4. Validate the ChromaDB collection

Read-only check that the Chapter 7 embedding pipeline's output (`chroma_db/`, checked into
the repo) is intact and queryable -- BGE-M3, 1024-dim vectors.

In [4]:
# Must run as a loose script from the REPO ROOT (not `cd src` first):
# validate_db.py imports storage.py with a bare `from storage import ...`,
# which only resolves because Python puts the script's own directory on
# sys.path -- but storage.py's CHROMA_PATH="./chroma_db" is resolved
# against the CURRENT WORKING DIRECTORY, which needs to stay the repo
# root for it to find the real (populated) chroma_db/ folder.
!python src/validate_db.py


DMRC ChromaDB Collection Validation
Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given
[OK] Connected to ChromaDB at ./chroma_db
[OK] Collection opened: 'dmrc_be12be14_ecs'
     Collection metadata : {'chunking_strategy': 'clause-level', 'embedding_model': 'BAAI/bge-m3'}

[OK] Total vectors stored : 63
Failed to send telemetry event CollectionGetEvent: capture() takes 1 positional argument but 3 were given

----------------------------------------------------------------------
Sample Record
----------------------------------------------------------------------
Chunk ID : DMRC-BE12BE14-VOL2-CH1-p10-c1.2.1-012

Sample document text (first 200 characters):
In a general manner, all works, facilities and services and other components as required whether or not specified necessary to deliver the requirements of this Specificat

### 5. Dense retrieval (`query.py`)

Straight vector search against ChromaDB, no BM25 or reranking yet.

In [5]:
!python -m src.query "Explain clause 1.2.1"


2026-07-24 06:47:55.117500: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1784875675.141898    5752 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1784875675.149610    5752 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1784875675.170050    5752 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1784875675.170086    5752 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1784875675.170090    5752 computation_placer.cc:177] computation placer alr

### 6. Hybrid retrieval (`hybrid_retriever.py`)

Dense (BGE-M3) + BM25, merged. Must be run with `-m` (relative imports inside the package),
not as a bare script.

In [6]:
!python -m src.hybrid_retriever "Explain clause 1.2.1"


2026-07-24 06:48:36.673373: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1784875716.699735    6007 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1784875716.707502    6007 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1784875716.729702    6007 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1784875716.729748    6007 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1784875716.729751    6007 computation_placer.cc:177] computation placer alr

### 7. Cross-encoder reranking (`reranker.py`)

In [7]:
!python -m src.reranker "What is the scope of work?"


2026-07-24 06:49:14.850331: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1784875754.873555    6222 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1784875754.881107    6222 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1784875754.900086    6222 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1784875754.900122    6222 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1784875754.900125    6222 computation_placer.cc:177] computation placer alr

### 8. End-to-end retrieval timing (no LLM)

Times each retrieval-side stage and estimates the prompt size the LLM would receive, so you
know *before* opening notebook 2 whether a query is going to produce a huge prompt.

In [8]:
import time
import src.hybrid_retriever as hr
import src.reranker as rr
import src.prompt_engineering as pe

QUERY = "What are the contractor obligations?"

t0 = time.time()
hits = hr.hybrid_search(QUERY)
t1 = time.time()
print(f"1. hybrid_search : {t1-t0:6.2f}s -> {len(hits)} candidates")

reranked = rr.rerank(QUERY, hits)
t2 = time.time()
print(f"2. rerank        : {t2-t1:6.2f}s -> {len(reranked)} kept")

prompt = pe.build_prompt(QUERY, reranked)
t3 = time.time()
print(f"3. build_prompt  : {t3-t2:6.2f}s")

approx_tok = len(prompt) // 4
print(f"\nEstimated prompt size: {len(prompt)} chars (~{approx_tok} tokens)")
if approx_tok > 6000:
    print("Very large -- notebook 2 applies retrieval caps (RAG_MAX_CANDIDATES / RAG_MAX_CONTEXT) for this.")
elif approx_tok > 3000:
    print("On the large side, but within what the caps in notebook 2 are tuned for.")
else:
    print("Reasonable size for the LLM stage.")


ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


Loading embedding model: BAAI/bge-m3 ...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given
ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given
ERROR:chromadb.telemetry.product

1. hybrid_search :   6.80s -> 5 candidates
Loading reranker model: BAAI/bge-reranker-v2-m3 ...
2. rerank        :   2.31s -> 5 kept
3. build_prompt  :   0.00s

Estimated prompt size: 4651 chars (~1162 tokens)
Reasonable size for the LLM stage.


---
### Next step

If every cell above ran without error, retrieval + reranking are verified end-to-end.
Switch this Colab runtime to **GPU**, and open **`02_Gemma_Inference_and_Serving.ipynb`**
to load Gemma 2 9B and serve `/ask` over the FastAPI app.